1. Given employer history data, where each record contains details about an employee’s work history — employer, job position, and the start and end dates of each job.

Solution : 
1. Sort the data
2. Use the LEAD function
3. Filter the results
4. Count the distinct users

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
spark = SparkSession.builder \
    .appName("LinkedInUsers") \
    .getOrCreate()

In [0]:
linkedin_data = [
    (1, 'Microsoft', 'developer', '2020-04-13', '2021-11-01'),
    (1, 'Google', 'developer', '2021-11-01', None),
    (2, 'Google', 'manager', '2021-01-01', '2021-01-11'),
    (2, 'Microsoft', 'manager', '2021-01-11', None),
    (3, 'Microsoft', 'analyst', '2019-03-15', '2020-07-24'),
    (3, 'Amazon', 'analyst', '2020-08-01', '2020-11-01'),
    (3, 'Google', 'senior analyst', '2020-11-01', '2021-03-04'),
    (4, 'Google', 'junior developer', '2018-06-01', '2021-11-01'),
    (4, 'Google', 'senior developer', '2021-11-01', None),
    (5, 'Microsoft', 'manager', '2017-09-26', None),
    (6, 'Google', 'CEO', '2015-10-02', None)
]

# Define the schema for the LinkedIn data
linkedin_columns = [
  'emp_id', 
  'employer', 
  'position', 
  'start_date', 
  'end_date'
]

In [0]:
linkedin_df = spark.createDataFrame(linkedin_data, linkedin_columns)

In [0]:
linkedin_df.printSchema()
linkedin_df.show()
#1. What is the total number of employees?
linkedin_df.count()
#2. What is the total number of unique employers?
linkedin_df.select('employer').distinct().count()
#3. What is the total number of unique positions?
linkedin_df.select('position').distinct().count()
#4. What is the total number of unique employers and positions

root
 |-- emp_id: long (nullable = true)
 |-- employer: string (nullable = true)
 |-- position: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)

+------+---------+----------------+----------+----------+
|emp_id| employer|        position|start_date|  end_date|
+------+---------+----------------+----------+----------+
|     1|Microsoft|       developer|2020-04-13|2021-11-01|
|     1|   Google|       developer|2021-11-01|      NULL|
|     2|   Google|         manager|2021-01-01|2021-01-11|
|     2|Microsoft|         manager|2021-01-11|      NULL|
|     3|Microsoft|         analyst|2019-03-15|2020-07-24|
|     3|   Amazon|         analyst|2020-08-01|2020-11-01|
|     3|   Google|  senior analyst|2020-11-01|2021-03-04|
|     4|   Google|junior developer|2018-06-01|2021-11-01|
|     4|   Google|senior developer|2021-11-01|      NULL|
|     5|Microsoft|         manager|2017-09-26|      NULL|
|     6|   Google|             CEO|2015-10

7

In [0]:
# Window spec definition
window_spec = Window.partitionBy('emp_id').orderBy('start_date')


In [0]:
# Use LEAD function to get the next employer for each user
linkedin_with_next_employer = linkedin_df.withColumn('next_employer', F.lead('employer').over(window_spec))


In [0]:
# Filter to find users who worked at Microsoft and then moved to Google
result = linkedin_with_next_employer.filter(
    (linkedin_with_next_employer.employer == 'Microsoft') & 
    (linkedin_with_next_employer.next_employer == 'Google')
)

In [0]:
# Select emp_id(s)
emp_ids = result.select('emp_id').distinct()

In [0]:
# Show the result
emp_ids.show()

+------+
|emp_id|
+------+
|     1|
+------+



In [0]:
# Count the number of distinct user_ids
user_count = emp_ids.count()
print(f"Number of users who worked at Microsoft and then moved to Google: {user_count}")

Number of users who worked at Microsoft and then moved to Google: 1
